# Planner-facing risk report (v1.2)

Reads the current best quantile forecast (`lifecycle_features` shared-horizon LightGBM quantile p50, WAPE 0.696590) and the Phase 7 scenario forecasts, and turns them into a per-product risk view aimed at planners.

**v1.2 polish** (this version):

- Dashboard now uses a single global footer; subplot footnotes don't overlap.
- Planning-example item is auto-selected from products with meaningful expected demand and meaningful risk buffer, not whatever has the widest band.
- High-uncertainty leaderboard excludes items with `expected_demand_p50 < demand_floor` by default (those belong on the low-expected/high-upside chart instead).
- Low-expected/high-upside chart now makes p50 (blue) vs p90 (purple) explicit in its title, legend, and footnote.

Headline metrics per product:

- `expected_demand_p50` -- model's median forecast summed across the planning window.
- `conservative_demand_p90` -- the conservative planning quantile, **not** a guarantee.
- `risk_buffer = p90 - p50` -- extra units needed to reach the p90 planning quantile.
- `stockout_attention_score = risk_buffer × log1p(p50)` -- volume-weighted ranking signal.
- `scenario_attention_score` -- analogous, predictive (not causal) scenario sensitivity.

All scenario numbers are predictive (model what-if), not causal.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
pd.options.display.float_format = '{:.4f}'.format

from seercast.training.generate_planner_report import run as run_planner
REPO_ROOT

## 1. Build the planner report

In [ ]:
result = run_planner()
risk = result['risk_report']
summary = result['summary']
summary

## 2. Top high-uncertainty products (top 10)

In [ ]:
(risk.sort_values('relative_uncertainty', ascending=False)
     [['id','cat_id','dept_id','expected_demand_p50','conservative_demand_p90',
       'risk_buffer','relative_uncertainty','uncertainty_label']]
     .head(10))

## 3. Top stockout-attention products (largest risk_buffer)

In [ ]:
(risk.sort_values('risk_buffer', ascending=False)
     [['id','cat_id','dept_id','expected_demand_p50','conservative_demand_p90',
       'risk_buffer','stockout_attention_label']]
     .head(10))

## 4. Top scenario-sensitive products

In [ ]:
cols = [c for c in ['id','cat_id','dept_id','max_abs_scenario_delta_p50_pct',
                    'most_sensitive_scenario','momentum_plus_delta_pct','momentum_minus_delta_pct',
                    'price_plus_delta_pct','price_minus_delta_pct','scenario_sensitivity_label']
        if c in risk.columns]
(risk.sort_values('max_abs_scenario_delta_p50_pct', ascending=False)
     [cols].head(10))

## 5. Generated figures

Saved to `outputs/figures/planner/`:

- `top_high_uncertainty_products.png`
- `top_stockout_attention_products.png`
- `top_scenario_sensitive_products.png`
- `item_planning_demand_example.png`
- `planner_dashboard_ca1.png`

**Reminders.**

- `p90` is the conservative planning quantile. It is not a guaranteed upper bound on demand.
- Scenario sensitivity is predictive only -- it tells you how the model's forecast moves under altered inputs. It does not estimate a causal effect.
- Risk labels (`high / medium / low`) are heuristic percentile bands (top 10% / next 20% / rest), not service-level targets.